In [1]:
import time
import json
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score
import xgboost as xgb

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [5]:
# Load German Credit dataset
df = pd.read_csv("german_credit_data.csv")

# Print all columns to verify exact header names
print("Columns in CSV:", df.columns.tolist())

# Flexibly locate target column: look for 'risk' or 'class', otherwise grab the last column
risk_cols = [col for col in df.columns if any(k in col.lower() for k in ["risk", "class", "target"])]

if risk_cols:
    target_col = risk_cols[0]
else:
    target_col = df.columns[-1]  # Default to last column

print(f"Selected target column: '{target_col}'")

# Map target values to binary (1 = Bad/High Risk, 0 = Good/Low Risk)
y = df[target_col].apply(lambda x: 1 if str(x).lower() in ["2", "bad", "1", "high"] else 0)
X = df.drop(target_col, axis=1)

# Apply One-Hot Encoding to categorical variables
X_encoded = pd.get_dummies(X, drop_first=True)

# Save feature names for UI matching
feature_names = list(X_encoded.columns)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)

# Fit Scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data preprocessed and scaled successfully!")

Columns in CSV: ['laufkont', 'laufzeit', 'moral', 'verw', 'hoehe', 'sparkont', 'beszeit', 'rate', 'famges', 'buerge', 'wohnzeit', 'verm', 'alter', 'weitkred', 'wohn', 'bishkred', 'beruf', 'pers', 'telef', 'gastarb', 'kredit']
Selected target column: 'kredit'
Data preprocessed and scaled successfully!


In [6]:
# Train XGBoost Classifier
xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train_scaled, y_train)

# Measure inference time and performance
start_time = time.time()
xgb_preds_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]
xgb_latency = (time.time() - start_time) / len(X_test_scaled) * 1000  # ms per sample

xgb_preds = (xgb_preds_proba > 0.5).astype(int)

xgb_metrics = {
    "auc": round(float(roc_auc_score(y_test, xgb_preds_proba)), 4),
    "precision": round(float(precision_score(y_test, xgb_preds)), 4),
    "recall": round(float(recall_score(y_test, xgb_preds)), 4),
    "latency_ms": round(float(xgb_latency), 3)
}
print("XGBoost Metrics:", xgb_metrics)

XGBoost Metrics: {'auc': 0.8198, 'precision': 0.8153, 'recall': 0.9143, 'latency_ms': 0.018}


In [7]:
# Convert data to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Define PyTorch Multi-Layer Perceptron
class CreditMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

mlp_model = CreditMLP(X_train_scaled.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(mlp_model.parameters(), lr=0.001)

# Training Loop
mlp_model.train()
for epoch in range(50):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        out = mlp_model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()

# PyTorch Inference Benchmarking
mlp_model.eval()
with torch.no_grad():
    start_time = time.time()
    mlp_preds_proba = mlp_model(X_test_tensor).numpy().flatten()
    mlp_latency = (time.time() - start_time) / len(X_test_scaled) * 1000

mlp_preds = (mlp_preds_proba > 0.5).astype(int)

mlp_metrics = {
    "auc": round(float(roc_auc_score(y_test, mlp_preds_proba)), 4),
    "precision": round(float(precision_score(y_test, mlp_preds)), 4),
    "recall": round(float(recall_score(y_test, mlp_preds)), 4),
    "latency_ms": round(float(mlp_latency), 3)
}
print("PyTorch MLP Metrics:", mlp_metrics)

PyTorch MLP Metrics: {'auc': 0.745, 'precision': 0.7917, 'recall': 0.8143, 'latency_ms': 0.007}


In [10]:
# Identify Top Feature Drivers
importances = xgb_model.feature_importances_
top_feature_indices = np.argsort(importances)[::-1][:3]
top_features = [feature_names[i] for i in top_feature_indices]

# Export Model, Scaler, and Feature Schema
joblib.dump(xgb_model, "xgboost_model.joblib")
joblib.dump(scaler, "scaler.joblib")

# Save exact feature layout so app.py aligns inputs properly
joblib.dump(feature_names, "feature_names.joblib")

# Export Summary Metrics JSON for Groq Agent
benchmark_summary = {
    "winning_model": "XGBoost",
    "metrics": {
        "XGBoost": xgb_metrics,
        "PyTorch_MLP": mlp_metrics
    },
    "top_features": top_features
}

with open("metrics.json", "w") as f:
    json.dump(benchmark_summary, f, indent=4)

print("Notebook execution complete. All artifacts saved!")

Notebook execution complete. All artifacts saved!
